# Airplane pipeline walkthrough

Demonstrates the airplane regression pipeline stage by stage using the
functional API. Run from the project root (`airplane/`).

In [ ]:
from helpers.config import get_config
from src.data.data_ingestion import fetch_raw_data
from src.data.feature_engineering import clean
from src.data.data_transformation import build_preprocessor, split

config = get_config()
config

## Ingest and clean

In [ ]:
raw = fetch_raw_data(config)
data = clean(raw, config)
data.head()

## Split and inspect the preprocessor

All transforms (log scaling, standardization) live inside this `ColumnTransformer`,
which is embedded in every candidate pipeline and persisted with the model.

In [ ]:
X_train, X_test, y_train, y_test = split(data, config)
preprocessor = build_preprocessor(X_train, config)
preprocessor

## Train, evaluate, predict

`train` logs every candidate to MLflow and persists the best pipeline; `evaluate`
scores it on the holdout split and writes `metrics.json` + `model_card.md`.

In [ ]:
from src.models.model_trainer import train
from src.models.model_evaluation import evaluate
from src.models.predict import predict

summary = train(X_train, X_test, y_train, y_test, preprocessor, config)
print('best model:', summary['best_model'])
print('holdout metrics:', evaluate(X_test, y_test, config))
print('sample prediction:', predict(X_test.iloc[0].to_dict(), config))